In [ ]:
config = {
  'model_id': 'ibm-granite/granite-embedding-107m-multilingual',
  'training_data': 'processed-spam-labels-merged.parquet.gzip',
  'token_limit': 512,
  'optim': "adamw_torch",
  'lr': 2e-5,
  'batch_size': 128,
  #'gradient_accumulation_steps': 8,
  'use_lora': True,
  'peft_r': 8,
  'lora_alpha': 32,
  'lora_dropout': 0.1,
  'strip_html': True
}

In [ ]:
metric_for_best_model = 'roc_auc'
epochs = 2000
checkpoint_name = "spam/" + "-".join(key + '=' + str(value)[:5].replace('/', ':') for key, value in config.items())

In [ ]:
!pip install -q transformers[torch]
!pip install -q datasets
!pip install -q sentencepiece
!pip install -q -U bitsandbytes
!pip install -q accelerate
!pip install -q peft
!pip install -q pandas
!pip install -q scikit-learn
!pip install -q protobuf
!pip install -q pyarrow
!pip install -q hf_transfer

In [ ]:
import os
import gc
import re
import torch
import numpy as np
import pandas as pd
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score, fbeta_score
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import EvalPrediction
from transformers import Trainer
from transformers import TrainingArguments
from datasets import Dataset

In [ ]:
# 執行前請在環境中設定 HF_TOKEN，例如 export HF_TOKEN=<your token>
os.environ.setdefault('HF_TOKEN', '')

In [ ]:
%%time
data = pd.read_parquet(config['training_data'])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config['model_id'])

In [ ]:
hf_dataset = Dataset.from_pandas(data)

def tokenize_function(example):
    tokens = tokenizer(
        example['text'],
        padding='max_length',
        truncation=True,
        max_length=config['token_limit']
    )
    tokens["labels"] = example["is_spam"]
    return tokens

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


In [ ]:
train_dataset, val_dataset = torch.utils.data.random_split(
    tokenized_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))

In [ ]:
tokenizer.decode(val_dataset[3]['input_ids'])

In [ ]:
val_dataset[3]['labels']

In [ ]:
_model = AutoModelForSequenceClassification.from_pretrained(
    config['model_id'],
    num_labels=2,
    cache_dir='.cache',
    torch_dtype='auto',
    trust_remote_code=True
)

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=config['peft_r'],
    lora_alpha=config['lora_alpha'],
    lora_dropout=config['lora_dropout'],
    modules_to_save=["score"]
)

In [ ]:
if config['use_lora']:
    model = get_peft_model(_model, peft_config)
    model.print_trainable_parameters()
else:
    model = _model

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
softmax = torch.nn.Softmax(dim=-1)

def spam_metrics(predictions, labels):
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    _probs = softmax(torch.Tensor(predictions))
    probs = _probs[:, -1]

    thresholds = [i * 0.01 for i in range(100)]
    precision = []
    recall = []
    f1 = []
    fbeta = []
    weighted_f1 = []
    accuracys = []
    for threshold in thresholds:
        y_pred = np.zeros(probs.shape)
        y_pred[np.where(probs >= threshold)] = 1
        y_true = labels
        precision.append(precision_score(y_true, y_pred))
        recall.append(recall_score(y_true, y_pred))
        f1.append(f1_score(y_true, y_pred))
        fbeta.append(fbeta_score(y_true, y_pred, beta=0.5))
        weighted_f1.append(f1_score(y_true, y_pred, average='weighted'))
        accuracys.append(accuracy_score(y_true, y_pred))

    idx = np.argmax(fbeta)
    return {
        'thold': thresholds[idx],
        'precision': precision[idx],
        'recall':  recall[idx],
        'f1': f1[idx],
        'fbeta_0.5': fbeta[idx],
        'weighted_f1': weighted_f1[idx],
        'accuracy': accuracys[idx],
        'roc_auc': roc_auc_score(y_true, probs)
    }

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    result = spam_metrics(predictions=preds, labels=p.label_ids)
    return result

In [ ]:
args = TrainingArguments(
    checkpoint_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim=config['optim'],
    learning_rate=config['lr'],
    per_device_train_batch_size=config['batch_size'],
    per_device_eval_batch_size=config['batch_size'],
    # gradient_accumulation_steps=config['gradient_accumulation_steps'],
    num_train_epochs=epochs,
    # weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_for_best_model,
)

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
#trainer.train(resume_from_checkpoint=True)
trainer.train()